# 📓 Notebook 8 — Valutazione dei modelli, Cross-Validation e Tuning degli iperparametri

Bentornato! 👋 Sei arrivato all'ultimo notebook del corso. Ottimo lavoro! 🎉

## 🎯 Cosa imparerai qui
1. Perché un singolo `train_test_split` non basta
2. **Cross-Validation (K-Fold)**: la tecnica corretta per valutare un modello
3. **GridSearchCV**: come trovare automaticamente i migliori iperparametri
4. **Pipeline**: come incatenare preprocessing + modello senza errori
5. Una **roadmap pratica** per continuare a imparare dopo questo corso

## 🤔 Perché questo notebook è importante
Finora abbiamo costruito modelli, ma li abbiamo valutati "al volo". In un progetto serio (uno stage, un lavoro, una competizione Kaggle) la valutazione **è la metà del lavoro**. Un modello mal valutato può sembrare bellissimo... e poi crollare nel mondo reale.

In [ ]:
# Import delle librerie che useremo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dataset
from sklearn.datasets import load_breast_cancer

# Modelli
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Strumenti di valutazione
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
print('Tutto pronto! ✅')

## 1. Il problema del singolo split

Nei notebook precedenti facevamo così:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
```

Il problema? **Il risultato dipende molto da come sono stati estratti quei dati di test.** Se sei fortunato hai un test "facile" e il modello sembra fortissimo. Se sei sfortunato hai un test "difficile" e il modello sembra scarso. Stesso modello, valutazioni diverse!
![problem](Media/08/problem.png)

Verifichiamolo concretamente. 👇

In [ ]:
# Carichiamo il dataset "breast cancer" (già usato nel notebook 4)
data = load_breast_cancer()
X, y = data.data, data.target

# Facciamo 10 split diversi (cambiando random_state) e vediamo l'accuratezza
accuratezze = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    modello = LogisticRegression(max_iter=10000)
    modello.fit(X_tr, y_tr)
    acc = modello.score(X_te, y_te)
    accuratezze.append(acc)
    print(f'Seed {seed}: accuratezza = {acc:.4f}')

print(f'\nMinimo: {min(accuratezze):.4f}')
print(f'Massimo: {max(accuratezze):.4f}')
print(f'Differenza: {max(accuratezze) - min(accuratezze):.4f}  ← ecco il problema!')

Vedi? Lo stesso modello, sugli stessi dati, può dare valori che variano di alcuni punti percentuali a seconda di come spezzi i dati. 😱

**Soluzione: Cross-Validation.**

## 2. K-Fold Cross-Validation

L'idea è semplice: invece di fare **un solo** split, ne facciamo **K** (di solito K=5 o K=10).

Esempio con K=5:
1. Dividiamo i dati in 5 parti uguali ("fold")
2. Alleniamo il modello su 4 parti, testiamo sulla 5ª → otteniamo `acc_1`
3. Alleniamo su altre 4 (cambiando quale tieni fuori), testiamo sulla rimanente → `acc_2`
4. ... ripetiamo finché ogni fold è stato "il test" una volta
5. La performance finale è la **media** delle 5 accuratezze (con la deviazione standard)

Così ogni esempio viene usato sia per allenare sia per testare, ma **mai contemporaneamente**. Risultato molto più robusto!

```
Fold 1:  [TEST ][train][train][train][train]
Fold 2:  [train][TEST ][train][train][train]
Fold 3:  [train][train][TEST ][train][train]
Fold 4:  [train][train][train][TEST ][train]
Fold 5:  [train][train][train][train][TEST ]
```
![kfold](Media/08/kfold.png)


In [ ]:
# Cross-validation con sklearn: una sola riga!
modello = LogisticRegression(max_iter=10000)

# cv=5 significa K=5 fold
scores = cross_val_score(modello, X, y, cv=5, scoring='accuracy')

print(f'Accuratezze sui 5 fold: {scores}')
print(f'Media: {scores.mean():.4f}')
print(f'Deviazione standard: {scores.std():.4f}')
print(f'\n→ Risultato finale: {scores.mean():.4f} ± {scores.std():.4f}')

💡 **Come si legge `0.95 ± 0.02`?**
Significa: "il modello ha circa il 95% di accuratezza, con un'oscillazione tipica del 2%". Molto più informativo di un singolo numero!

### StratifiedKFold (per classificazione sbilanciata)
Se hai un dataset sbilanciato (es. 90% classe A, 10% classe B), il normale K-Fold potrebbe pescare un fold con 0 esempi della classe B! La **stratificazione** assicura che ogni fold mantenga le stesse proporzioni dell'intero dataset.

In [ ]:
# Cross-validation stratificata (consigliata SEMPRE per la classificazione)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(modello, X, y, cv=skf, scoring='accuracy')
print(f'Accuratezze stratificate: {scores}')
print(f'Media: {scores.mean():.4f} ± {scores.std():.4f}')

### 🔧 PROVA TU
Confronta tre modelli diversi tramite cross-validation. Chi vince?

In [ ]:
# Confronto fra 3 modelli con la stessa cross-validation
modelli = {
    'Logistic Regression': LogisticRegression(max_iter=10000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC()
}

for nome, m in modelli.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    print(f'{nome:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Iperparametri: cosa sono?

Quando crei un modello, ci sono **due tipi di parametri**:

| Tipo | Chi li sceglie | Esempio |
|------|----------------|---------|
| **Parametri** | Il modello li impara dai dati durante `fit()` | i pesi `w` di una regressione lineare |
| **Iperparametri** | **TU** li scegli prima di chiamare `fit()` | `max_depth` di un albero, `C` di una SVM |

Gli iperparametri controllano *come* il modello impara. Sceglierli male può fare la differenza tra un modello eccezionale e uno mediocre.

**Domanda:** come trovo i valori migliori? Provandoli tutti! Ma in modo intelligente. → **GridSearchCV**.

## 4. GridSearchCV: la ricerca automatica

L'idea: dichiari quali iperparametri vuoi provare e con quali valori, e sklearn:
1. Prova **tutte** le combinazioni
2. Per ognuna fa cross-validation
3. Ti dice qual è la combinazione migliore

Esempio: per un Random Forest vogliamo provare diversi `n_estimators` e `max_depth`.


![grid](Media/08/grid.png)


In [ ]:
# Definiamo lo "spazio di ricerca": dizionario { iperparametro: [valori da provare] }
param_grid = {
    'n_estimators': [50, 100, 200],     # numero di alberi nella foresta
    'max_depth': [3, 5, 10, None],      # profondità max (None = nessun limite)
    'min_samples_split': [2, 5]          # campioni minimi per dividere un nodo
}
# Combinazioni totali: 3 * 4 * 2 = 24
# Con cv=5 → 24 * 5 = 120 modelli allenati! Sklearn li gestisce per noi 🚀

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    estimator=rf,                  # quale modello
    param_grid=param_grid,         # quali combinazioni
    cv=5,                          # 5-fold cross-validation
    scoring='accuracy',            # metrica da ottimizzare
    n_jobs=-1                      # -1 = usa tutti i core CPU disponibili
)

grid.fit(X, y)

print(f'Migliori iperparametri: {grid.best_params_}')
print(f'Miglior accuratezza:    {grid.best_score_:.4f}')

In [ ]:
# Possiamo guardare TUTTI i risultati ordinati
risultati = pd.DataFrame(grid.cv_results_)
colonne = ['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 'mean_test_score', 'std_test_score']
risultati[colonne].sort_values('mean_test_score', ascending=False).head(10)

💡 **Il modello finale è già pronto!** Dopo `grid.fit()`, sklearn ri-allena automaticamente il modello migliore su **tutti** i dati. Lo trovi in `grid.best_estimator_`.

### ⚠️ Attenzione al data leakage
Se devi normalizzare i dati (StandardScaler) e fare cross-validation, NON normalizzare prima! Altrimenti il modello "vede" il test set durante l'allenamento (data leakage). La soluzione è la **Pipeline**.

## 5. Pipeline: incatenare preprocessing + modello

Una `Pipeline` è una catena di passaggi che si applicano in ordine. Il vantaggio? La cross-validation sa che lo scaling va fatto **dentro** ogni fold, non prima.

![pipeline](Media/08/pipeline.png)


In [ ]:
# Pipeline: prima scala i dati, poi allena la SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),     # passo 1: normalizza
    ('svm', SVC())                     # passo 2: classifica
])

# Per la grid search, i nomi degli iperparametri sono: nome_passo__nome_iperparametro
param_grid = {
    'svm__C': [0.1, 1, 10, 100],         # parametro di regolarizzazione
    'svm__kernel': ['linear', 'rbf'],     # tipo di kernel
    'svm__gamma': ['scale', 'auto']        # parametro del kernel rbf
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X, y)

print(f'Migliore configurazione: {grid.best_params_}')
print(f'Accuratezza:             {grid.best_score_:.4f}')

## 6. Workflow completo: il modo "giusto" di fare ML
![wf](Media/08/wf.png)

Ecco lo schema che dovresti applicare in QUALSIASI progetto ML:

In [ ]:
# ────────────────────────────────────────────────────────────────
# WORKFLOW COMPLETO (riferimento per i tuoi progetti futuri)
# ────────────────────────────────────────────────────────────────

# STEP 1: Separa SUBITO un test set finale ("non lo guardiamo finché alla fine")
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# STEP 2: Costruisci la pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=10000))
])

# STEP 3: Trova i migliori iperparametri SOLO sul dev set (con cross-validation)
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100]    # regolarizzazione
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_dev, y_dev)

print(f'Migliore C: {grid.best_params_}')
print(f'Accuratezza CV: {grid.best_score_:.4f}')

# STEP 4: Valuta sul test set (UNA SOLA VOLTA!)
miglior_modello = grid.best_estimator_
predizioni = miglior_modello.predict(X_test)
print(f'\nAccuratezza sul test set finale: {accuracy_score(y_test, predizioni):.4f}')
print('\nReport completo:')
print(classification_report(y_test, predizioni, target_names=data.target_names))

💡 **Perché tenere un test set "intoccato"?**
Se usi i dati di test per scegliere il modello migliore, alla fine non sai più se quel numero è onesto. Il test set finale è la tua "prova del nove" - lo guardi UNA volta sola, alla fine.

Schema mentale:
```
Tutti i dati
    ├── Train+Validation (80%) ← qui faccio cross-validation e tuning
    └── Test (20%)              ← guardato UNA VOLTA, alla fine
```

## 7. Quale metrica scegliere?

Una checklist veloce:

| Tipo di problema | Metrica suggerita | Quando |
|------------------|-------------------|--------|
| Classificazione bilanciata | `accuracy` | Le classi sono ~50/50 |
| Classificazione sbilanciata | `f1`, `roc_auc` | Una classe rara (es. frodi) |
| Frodi/diagnosi mediche | `recall` | Meglio falsi allarmi che mancare un caso |
| Spam/raccomandazioni | `precision` | Meglio mancarne uno che dare un falso positivo |
| Regressione | `neg_root_mean_squared_error`, `r2` | A seconda di cosa vuoi minimizzare |

Cambia `scoring='...'` in `cross_val_score` o `GridSearchCV` per usare un'altra metrica.

![error](Media/08/error.png)


## 🎓 Riepilogo del corso completo

Hai percorso un viaggio importante! Ecco cosa hai imparato:

| # | Notebook | Concetti chiave |
|---|----------|----------------|
| 1 | Matematica base | Vettori, matrici, derivate, gradiente, statistica |
| 2 | Dati | Pandas, pulizia, encoding, scaling, split |
| 3 | Regressione lineare | MSE, gradient descent, R² |
| 4 | Regressione logistica | Sigmoide, soglia, precision/recall/F1, ROC |
| 5 | Alberi e Random Forest | Pruning, ensemble, feature importance |
| 6 | K-Means | Clustering non supervisionato, gomito, silhouette |
| 7 | PCA | Riduzione dimensionalità, varianza spiegata |
| 8 | Valutazione e tuning | CV, GridSearch, Pipeline, workflow |

## 🗺️ Roadmap: dove andare adesso

### Livello successivo (subito dopo questo corso):
1. **Pratica su Kaggle**: prova competizioni "Getting Started" come *Titanic* o *House Prices*
2. **Approfondisci sklearn**: esplora altri modelli — Gradient Boosting, XGBoost, LightGBM
3. **Feature engineering**: l'arte di creare nuove colonne dai dati esistenti (spesso più importante del modello!)
4. **Imbalanced learning**: SMOTE, class_weight per classi sbilanciate

### Strada del Deep Learning:
1. **Reti neurali base** con PyTorch o Keras/TensorFlow
2. **CNN** per immagini (computer vision)
3. **Transformer** per testo (NLP, LLM)

### Strada "applicazioni reali":
1. **MLOps**: come mettere un modello in produzione (Docker, FastAPI, MLflow)
2. **Time series**: previsioni temporali (ARIMA, Prophet)
3. **Recommender systems**: come fa Netflix/Spotify

### Risorse consigliate (gratuite):
- 📚 *"Hands-On Machine Learning"* di Aurélien Géron (libro - il più consigliato in assoluto)
- 🎥 *Andrew Ng - Machine Learning Specialization* su Coursera
- 🎥 *StatQuest* su YouTube (spiegazioni intuitive eccezionali)
- 📖 La documentazione di sklearn: scikit-learn.org/stable/user_guide.html
- 🏆 Kaggle Learn: kaggle.com/learn (mini-corsi gratis, molto pratici)

## 💡 Consiglio finale

Il ML si impara **con le mani sporche**. Scegli un dataset che ti interessa davvero (sport, musica, videogiochi, finanza, ...) e prova ad applicare quello che hai imparato. I primi 3-4 progetti saranno frustranti, poi qualcosa scatta. 🚀

**In bocca al lupo per il tuo viaggio nel ML! 🍀**